In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import random, os, math
import tensorflow as tf
import tensorflow_hub as hub
import tf_keras as keras
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()

In [ ]:
OPENAI_KEY = os.getenv('OPENAI_KEY')
BASE_URL = "http://192.168.1.186:11434/v1"

In [ ]:
client = OpenAI(api_key=OPENAI_KEY,base_url=BASE_URL)

In [ ]:
def get_embeddings(text, model="embeddinggemma"):
    text = text.replace("\n"," ")
    return client.embeddings.create(input=text, model=model).data[0].embedding

In [ ]:
dataset = []
with open('the_hunger_games.txt', 'r') as file:
  dataset = file.readlines()
  print(f'Loaded {len(dataset)} entries')

In [ ]:
VECTOR_DB = []

def add_chunk_to_database(chunk):
  embedding = get_embeddings(chunk)
  VECTOR_DB.append((chunk, embedding))


In [ ]:
for i, chunk in enumerate(dataset):
  if i > 999:
    break
  add_chunk_to_database(chunk)
  print(f'Added chunk {i+1}/{len(dataset)} to the database')

In [ ]:
def cosine_similarity(vec1, vec2):
  dot_product = sum([x * y for x, y in zip(vec1, vec2)])
  norm_a = sum([x ** 2 for x in vec1]) ** 0.5
  norm_b = sum([x ** 2 for x in vec2]) ** 0.5
  return dot_product / (norm_a * norm_b)


In [ ]:
def retrieve(query, top_n=3):
  query_embedding = get_embeddings(query)
  similarities = []
  for chunk, embedding in VECTOR_DB:
    similarity = cosine_similarity(query_embedding, embedding)
    similarities.append((chunk, similarity))
  similarities.sort(key=lambda x: x[1], reverse=True)
  return similarities[:top_n]


In [ ]:
vec1 = get_embeddings("This is not a test")
vec2 = get_embeddings("There are many stars in the sky")
print(cosine_similarity(vec1,vec2))

In [ ]:
def ask_gpt(system_prompt, user_prompt, model='gemma3:12b', temp=0.7):
  temperature=temp
  completion = client.chat.completions.create(
      model=model,
      temperature=temperature,
      messages=[
          {"role":"system",
          "content":system_prompt},
          {"role":"user",
           "content":user_prompt}])
  return completion.choices[0].message.content, completion

In [ ]:
input_query = "How are tributes, name any one?"
retrieved_knowledge = retrieve(input_query,40)
print('Retrieved knowledge:')
for chunk, similarity in retrieved_knowledge:
  print(f' - (similarity: {similarity:.2f}) {chunk}')
instruction_prompt = f'''You are a helpful chatbot.
Use only the following pieces of context to answer the question. Don't make up any new information:
{'\n'.join([f' - {chunk}' for chunk, similarity in retrieved_knowledge])}
'''
print(instruction_prompt)


In [ ]:
text, comp= ask_gpt(instruction_prompt, input_query)
print(text)